In [2]:
import pandas as pd
df = pd.read_csv("/content/IMDB Dataset.csv", encoding='latin1')

print(df.head())
print(df['sentiment'].value_counts())

df


df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

df


import re
negation_words = ["not good", "not bad", "not great", "don't like", "no good"]

def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # convert negation into single tokens
    for phrase in negation_words:
        text = text.replace(phrase, phrase.replace(" ", "_"))

    return text

df['review'] = df['review'].apply(clean_text)


from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<00v>")
tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

x_train_pad = pad_sequences(x_train_seq, maxlen=max_len, padding='post')
x_test_pad = pad_sequences(x_test_seq, maxlen=max_len, padding='post')


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),

    LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    x_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

loss, acc = model.evaluate(x_test_pad, y_test)
print("Test Accuracy:", acc)


def predict_sentiment(review):
    review = clean_text(review)
    seq = tokenizer.texts_to_sequences([review])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    prediction = model.predict(padded)[0][0]

    print("\nreview:", review)
    print("score:", prediction)

    if prediction >= 0.5:
        print("positive 😊")
    else:
        print("negative 😒")


predict_sentiment("this movie was not great")

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 369s 729ms/step - accuracy: 0.5493 - loss: 0.6716 - val_accuracy: 0.5756 - val_loss: 0.6497
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 381s 727ms/step - accuracy: 0.5910 - loss: 0.6223 - val_accuracy: 0.5763 - val_loss: 0.6483
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 381s 725ms/step - accuracy: 0.6702 - loss: 0.5545 - val_accuracy: 0.8332 - val_loss: 0.4250
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 367s 734ms/step - accuracy: 0.8594 - loss: 0.3517 - val_accuracy: 0.8670 - val_loss: 0.3792
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 362s 724ms/step - accuracy: 0.9111 - loss: 0.2397 - val_accuracy: 0.8709 - val_loss: 0.3618
313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 96ms/step - accuracy: 0.8733 - loss: 0.3546
Test Accuracy: 0.8733000159263611
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step

review: this movie was not_great
score: 0.61377835
positive 😊
